# Vehicle Failure Management — Condition Model Training



In [1]:
# Upload train_condition.csv and test_condition.csv
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving test_condition.csv to test_condition.csv
Saving train_condition.csv to train_condition.csv
Uploaded files: ['test_condition.csv', 'train_condition.csv']


In [2]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score)

TRAIN_PATH, TEST_PATH = Path('train_condition.csv'), Path('test_condition.csv')
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Upload both CSV files using their exact filenames.')
train, test = pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)
FEATURES = ['engine_coolant_temperature','oil_pressure_kpa','engine_rpm','vehicle_speed','throttle_position','engine_load']
TARGET, GROUP = 'condition', 'sequence_id'
LABELS = ['normal','warning','critical']
required = FEATURES + [TARGET, GROUP]
for name, df in [('train', train), ('test', test)]:
    missing = sorted(set(required) - set(df.columns))
    if missing: raise ValueError(f'{name} missing columns: {missing}')
    if df[required].isna().any().any(): raise ValueError(f'{name} has missing values in required fields')
    if set(df[TARGET].unique()) != set(LABELS): raise ValueError(f'{name} unexpected labels: {df[TARGET].unique()}')
if not set(train[GROUP]).isdisjoint(set(test[GROUP])):
    raise ValueError('Sequence leakage: some sequence_id values occur in both sets')
print('Train rows/sequences:', len(train), train[GROUP].nunique())
print('Test rows/sequences:', len(test), test[GROUP].nunique())
print('Sequence overlap:', len(set(train[GROUP]) & set(test[GROUP])))
print('Train label counts:\n', train[TARGET].value_counts())
print('Test label counts:\n', test[TARGET].value_counts())

Train rows/sequences: 40000 400
Test rows/sequences: 10000 100
Sequence overlap: 0
Train label counts:
 condition
normal      30782
critical     5378
warning      3840
Name: count, dtype: int64
Test label counts:
 condition
normal      7517
critical    1597
warning      886
Name: count, dtype: int64


In [3]:
OUT = Path('model_training_outputs')
OUT.mkdir(exist_ok=True)
X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train[TARGET], test[TARGET]
label_to_int = {label: i for i, label in enumerate(LABELS)}
y_train_int = y_train.map(label_to_int)
models = {
    'random_forest': RandomForestClassifier(n_estimators=400, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'xgboost': XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=400, max_depth=6,
        learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=42,
        n_jobs=-1, eval_metric='mlogloss', tree_method='hist')
}
summary, reports = [], {}
for name, model in models.items():
    print('Training', name)
    model.fit(X_train, y_train if name == 'random_forest' else y_train_int)
    raw_pred = model.predict(X_test)
    if name == 'xgboost':
        pred = pd.Series(raw_pred).map({i: label for label, i in label_to_int.items()}).to_numpy()
        prob_columns = LABELS
    else:
        pred = raw_pred
        prob_columns = list(model.classes_)
    probs = pd.DataFrame(model.predict_proba(X_test), columns=prob_columns)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, labels=LABELS, average='macro', zero_division=0)
    auc_labels = sorted(LABELS)
    auc = roc_auc_score(y_test, probs[auc_labels].to_numpy(), labels=auc_labels, multi_class='ovr', average='macro')
    row = {'model': name, 'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)), 'macro_precision': float(precision),
        'macro_recall': float(recall), 'macro_f1': float(f1), 'macro_roc_auc_ovr': float(auc)}
    summary.append(row)
    reports[name] = classification_report(y_test, pred, labels=LABELS, zero_division=0)
    pd.DataFrame(confusion_matrix(y_test, pred, labels=LABELS), index=[f'true_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS]).to_csv(OUT/f'{name}_confusion_matrix.csv')
    pd.DataFrame({'y_true': y_test.to_numpy(), 'y_pred': pred}).to_csv(OUT/f'{name}_test_predictions.csv', index=False)
    pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance', ascending=False).to_csv(OUT/f'{name}_feature_importance.csv', index=False)
    joblib.dump(model, OUT/f'{name}_condition_model.joblib')
    print(row)
    print(classification_report(y_test, pred, labels=LABELS, zero_division=0))
pd.DataFrame(summary).to_csv(OUT/'model_comparison_metrics.csv', index=False)
with open(OUT/'classification_reports.json', 'w') as f: json.dump(reports, f, indent=2)
with open(OUT/'training_summary.json', 'w') as f: json.dump({'target': TARGET, 'features': FEATURES, 'labels': LABELS,
    'train_rows': len(train), 'test_rows': len(test), 'train_sequences': int(train[GROUP].nunique()),
    'test_sequences': int(test[GROUP].nunique()), 'sequence_overlap': len(set(train[GROUP]) & set(test[GROUP])),
    'metrics': summary, 'interpretation_warning': 'Synthetic-data performance only; not evidence of real-world Toyota Aqua diagnostic accuracy.'}, f, indent=2)
print('Saved outputs to', OUT)

Training random_forest
{'model': 'random_forest', 'accuracy': 0.9998, 'balanced_accuracy': 0.999746931376336, 'macro_precision': 0.9995798574965016, 'macro_recall': 0.999746931376336, 'macro_f1': 0.9996632557735231, 'macro_roc_auc_ovr': 0.9999999230023229}
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      7517
     warning       1.00      1.00      1.00       886
    critical       1.00      1.00      1.00      1597

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

Training xgboost
{'model': 'xgboost', 'accuracy': 0.9969, 'balanced_accuracy': 0.9918358505999821, 'macro_precision': 0.9913485220195257, 'macro_recall': 0.9918358505999821, 'macro_f1': 0.9915912498376152, 'macro_roc_auc_ovr': 0.9999709542853946}
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00      7517
     w

In [4]:
# Download all metrics, plots/tables, and saved models as one ZIP
import shutil
from google.colab import files
zip_path = shutil.make_archive('model_training_outputs', 'zip', root_dir='model_training_outputs')
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>